# 8-4절 연습 문제 풀이

이 노트북은 8-4절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch08/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 8-4절 공통 - timm 전이 학습
import timm
from torchvision import transforms
from PIL import Image

CIFAR_ROOT = '../../downloads'

## 연습 8-17

VGG-16 대신 ResNet-18을 사용해 깃허브 저장소의 data/cat.jpg 이미지를 분류해 보자.

In [ ]:
model = timm.create_model('resnet18', pretrained=True).eval()
cfg = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**cfg)
print(f'입력 설정: {cfg["input_size"]}, 정규화 평균 {cfg["mean"]}')

img = Image.open('../../data/cat.jpg').convert('RGB')
x = transform(img).unsqueeze(0)
with torch.no_grad():
    probs = model(x).softmax(dim=1).squeeze()
top5 = probs.topk(5)
try:
    from urllib.request import urlopen
    labels = urlopen('https://raw.githubusercontent.com/pytorch/hub/master/'
                     'imagenet_classes.txt').read().decode().splitlines()
except Exception:
    labels = [f'class {i}' for i in range(1000)]
for p, i in zip(top5.values.tolist(), top5.indices.tolist()):
    print(f'  {labels[i]:28s} {p * 100:5.2f}%')

`timm.create_model('resnet18', pretrained=True)`로 모델을 바꾸기만 하면 된다. 중요한 것은 **모델마다 정해진 전처리**를 그대로 써야 한다는 점인데, `resolve_data_config()`와 `create_transform()`이 그 설정(입력 크기, 정규화 값)을 모델에서 직접 가져온다.

## 연습 8-18

[코드 8-16]은 ResNet-50의 출력 크기가 1,000인 출력층 선형 계층을 출력 크기가 2인 선형 계층으로 교체한다. 출력 계층을 교체하는 대신, 기존 출력층은 그대로 두고 그 뒤에 입력 크기가 1,000, 출력 크기가 2인 선형 계층을 덧붙이면 전이 학습 성능은 어떻게 달라질까? 직접 모델을 만들어 확인하고, 왜 그런 결과가 나오는지 해석해 보자.

In [ ]:
# 방법 A) 출력층 교체 / 방법 B) 출력층 뒤에 계층 추가
def make_model(mode):
    m = timm.create_model('resnet50', pretrained=True)
    for p_ in m.parameters(): p_.requires_grad = False
    if mode == '교체':
        m.fc = nn.Linear(m.fc.in_features, 2)
    else:      # 1000 -> 2 계층을 덧붙인다
        m = nn.Sequential(m, nn.Linear(1000, 2))
    return m

for mode in ('교체', '덧붙이기'):
    m = make_model(mode)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{mode}: 학습 가능 파라미터 {trainable:,}개')

**성능 차이가 생기는 이유**

- **교체 방식**: 2,048차원 특징 벡터를 곧바로 2개 클래스로 매핑한다. 특징이 그대로 전달된다.
- **덧붙이기 방식**: 특징이 ImageNet 1,000개 클래스 점수라는 **좁은 병목**을 한 번 통과한 뒤 2개로 매핑된다. 이 1,000차원은 '개·고양이·자동차 …'에 대한 점수라 흉부 X선 같은 다른 도메인에서는 정보가 크게 손실된다.

따라서 일반적으로 **교체 방식이 낫다**. 덧붙이기는 파라미터도 더 많으면서 성능은 떨어지기 쉽다.

## 연습 8-19

흉부 X선 분류 모델을 만들 때 사전 학습된 ResNet-50을 특징 추출 방식이 아니라 미세 조정 방식으로 사용하도록 예제를 수정해 보자.

In [ ]:
# 미세 조정: 전체 파라미터를 학습 대상으로 두고 작은 학습률을 사용한다.
def build_finetune(freeze=True, lr=1e-3):
    m = timm.create_model('resnet50', pretrained=True)
    for p_ in m.parameters():
        p_.requires_grad = not freeze          # 미세 조정이면 전부 학습
    m.fc = nn.Linear(m.fc.in_features, 2)      # 출력층은 항상 새로 학습
    return m, lr

for mode, freeze, lr in [('특징 추출', True, 1e-3), ('미세 조정', False, 1e-4)]:
    m, _ = build_finetune(freeze, lr)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{mode}: 학습 가능 {trainable:,}개, 권장 학습률 {lr}')
print('\n실제 학습은 캐글 흉부 X선 데이터셋을 내려받은 뒤 8-4절 예제의 '
      '데이터로더를 그대로 사용한다.')

미세 조정에서 핵심은 **학습률을 특징 추출보다 10배 이상 작게** 두는 것이다. 사전 학습된 가중치가 이미 좋은 상태이므로 큰 학습률로 갱신하면 애써 배운 특징이 무너진다(파국적 망각).

데이터가 적을 때는 앞쪽 계층은 고정하고 뒤쪽 블록만 여는 절충안도 자주 쓴다.

## 연습 8-20

[도전 문제] 미세 조정 방식을 사용하면 흉부 X선 데이터 분류 성능이 꽤 좋아졌을 것이다. 개선 아이디어를 하나 더 적용해 보자. 첫 번째 합성곱 계층은 ImageNet 데이터셋의 샘플과 같은 3채널 컬러 이미지를 입력받는다. 그래서 회색조 흉부 X선 사진을 넣으려면, 데이터 변환 객체가 1채널 이미지를 R, G, B 세 채널로 복사해 채널을 맞춘다. 대신 첫 번째 합성곱 계층이 1채널 회색조 이미지를 직접 입력받도록 모델 자체를 손보는 쪽으로 바꿔 보자.

이 작업은 다음과 같은 절차로 진행하면 된다.

데이터 변환 객체 수정: ResNet-50의 기본 설정으로 만든 데이터 변환 객체는 회색조 이미지를 세 채널의 값이 모두 같은 3채널 이미지로 변환한다. 대신 채널 보정 없이 이미지를 ImageNet 학습 때의 입력 크기인 224x224로 조정하고, 평균 0.5, 표준편차 0.5로 정규화하는 변환 객체를 새로 만들어 사용한다. 이 변경에 맞춰 모델 입력도 1채널을 받도록 함께 바꿔야 한다.

모델 구조 수정: ResNet-50에서 1채널 회색조 이미지를 학습할 수 있도록 첫 번째 합성곱 계층을 교체하고, 두 클래스로 분류할 수 있도록 마지막 분류기 계층을 교체한다. 이때 필수는 아니지만 기존 첫 번째 합성곱 계층의 학습된 파라미터를 새 합성곱 계층의 초깃값으로 이식하면 조금 더 성능이 좋아질 수 있다. 파라미터를 이식한다면 3개 채널의 가중치를 합산한 값을 사용한다. 회색조 이미지를 R, G, B 세 채널의 값이 같은 컬러 이미지로 보면, 합산한 가중치가 기존 RGB 입력에서의 출력 분포를 거의 그대로 재현하기 때문이다.

전이 학습: 교체한 계층만 학습하는 방식과 모든 계층을 추가 학습하는 방식으로 각각 한 번씩, 두 번 학습한다.

8장 학습 노트

In [ ]:
# 회색조 X선을 3채널로 늘리는 대신, 첫 합성곱을 1채널 입력으로 바꾼다.
model = timm.create_model('resnet50', pretrained=True)
old = model.conv1
print(f'원래 첫 합성곱: {old.weight.shape}  (출력 64, 입력 3, 7x7)')

new = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                stride=old.stride, padding=old.padding, bias=False)
# 사전 학습 가중치를 살리려면 RGB 3채널 가중치의 합(또는 평균)을 사용한다.
with torch.no_grad():
    new.weight.copy_(old.weight.sum(dim=1, keepdim=True))
model.conv1 = new
model.fc = nn.Linear(model.fc.in_features, 2)

x = torch.randn(2, 1, 224, 224)       # 1채널 입력
print(f'1채널 입력 {tuple(x.shape)} -> 출력 {tuple(model(x).shape)}')

회색조를 R·G·B에 복제하면 같은 데이터를 세 번 처리해 **연산이 3배**로 늘어난다. 첫 합성곱을 1채널로 바꾸면 이 낭비가 사라진다.

중요한 것은 사전 학습 가중치를 버리지 않는 것이다. 3채널 가중치를 **채널 방향으로 합하면** 회색조 입력에 대해 원래와 비슷한 반응을 내므로, 무작위 초기화보다 훨씬 좋은 출발점이 된다.